In [ ]:
"""
PIPELINE D'INTÉGRATION FINALE — VERSION CORRIGÉE [Bug #2 + #5 + #6]
=====================================================================
CORRECTIONS APPLIQUÉES :

  BUG #2 — CSV relu à chaque image (pd.read_csv dans analyser_radio)
           → AVANT : 500 images Phase 4 = 500 lectures du CSV NIH (112k lignes)
           → APRÈS : CSV chargé UNE SEULE FOIS en dehors de la boucle
           → Nouveau paramètre : df_meta (DataFrame indexé) au lieu de csv_path
           → Compatibilité maintenue : csv_path toujours accepté pour usage simple

  BUG #5 — CTR non calibré sans CSV non signalé
           → Ajout de ctr_warning affiché dans le rapport console

  BUG #6 — Transforms t512/t224 recréés à chaque appel
           → Déplacés en constantes globales T512 / T224
=====================================================================
"""
import os
import cv2
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torchvision import transforms, models
from PIL import Image

# ============================================================
# CHEMINS
# ============================================================
MODELS_DIR = '/kaggle/input/datasets/abdallahouahmane/pfe-cardio-models'
CSV_PATH   = '/kaggle/input/datasets/rahulogoel/clahe-enhancement-on-chestx-ray14/Data_Entry.csv'
IMG_DIR    = '/kaggle/input/datasets/rahulogoel/clahe-enhancement-on-chestx-ray14/IMAGE/IMAGE'

UNET_CARDIO  = f'{MODELS_DIR}/unet_cardio_robust_best.pth'
UNET_CALCIF  = f'{MODELS_DIR}/unet_calcif_best_kagg (2).pth'
CALCIF_MODEL = f'{MODELS_DIR}/calcif_model_best.pth'
RISK_MODEL   = f'{MODELS_DIR}/cardio_risk_v2_best (1).pth'

CATEGORIES_RISK   = ['Normal', 'Cardiomegaly', 'Edema']
CATEGORIES_CALCIF = ['Normal', 'Calcification', 'Aortic_enlargement']

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

# ============================================================
# BUG #6 CORRIGÉ — Transforms en constantes globales
# (avant : recréés à chaque appel de analyser_radio)
# ============================================================
T512 = transforms.Compose([
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])
T224 = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


# ============================================================
# 1. ARCHITECTURES — INCHANGÉES
# ============================================================
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super(UNet, self).__init__()
        def double_conv(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c), nn.ReLU(inplace=True)
            )
        self.enc1 = double_conv(in_channels, 64)
        self.enc2 = double_conv(64, 128)
        self.enc3 = double_conv(128, 256)
        self.enc4 = double_conv(256, 512)
        self.pool = nn.MaxPool2d(2, 2)
        self.bottleneck = double_conv(512, 1024)
        self.up4  = nn.ConvTranspose2d(1024, 512, 2, 2)
        self.dec4 = double_conv(1024, 512)
        self.up3  = nn.ConvTranspose2d(512, 256, 2, 2)
        self.dec3 = double_conv(512, 256)
        self.up2  = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec2 = double_conv(256, 128)
        self.up1  = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec1 = double_conv(128, 64)
        self.final = nn.Conv2d(64, out_channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([e4, self.up4(b)],  dim=1))
        d3 = self.dec3(torch.cat([e3, self.up3(d4)], dim=1))
        d2 = self.dec2(torch.cat([e2, self.up2(d3)], dim=1))
        d1 = self.dec1(torch.cat([e1, self.up1(d2)], dim=1))
        return torch.sigmoid(self.final(d1))


class EfficientNetCNN(nn.Module):
    def __init__(self, num_classes=3):
        super(EfficientNetCNN, self).__init__()
        self.backbone = models.efficientnet_b0(weights=None)
        orig = self.backbone.features[0][0]
        self.backbone.features[0][0] = nn.Conv2d(
            1, orig.out_channels,
            kernel_size=orig.kernel_size,
            stride=orig.stride,
            padding=orig.padding,
            bias=False
        )
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(in_features, 256),
            nn.ReLU(), nn.Dropout(0.2), nn.Linear(256, num_classes)
        )
        self._activations = None
        self.backbone.features[-1].register_forward_hook(self._save_act)

    def _save_act(self, m, i, o):
        self._activations = o

    def forward(self, x):
        return self.backbone(x)

    def get_gradcam(self, inp):
        self.eval()
        inp = inp.clone().requires_grad_(True)
        self._activations = None
        out = self.backbone(inp)
        idx = out.argmax(dim=1).item()
        self.zero_grad()
        out[0, idx].backward()
        if self._activations is None:
            return None, out
        act     = self._activations.detach().cpu().numpy()[0]
        weights = act.mean(axis=(1, 2))
        cam     = sum(w * a for w, a in zip(weights, act))
        cam     = cam.clip(min=0)
        if cam.max() > 0:
            cam /= cam.max()
        return cam, out


# ============================================================
# 2. CHARGEMENT DES MODÈLES — INCHANGÉ
# ============================================================
def charger_modeles():
    print("Chargement des 4 modèles...")

    m1 = UNet().to(device)
    m1.load_state_dict(torch.load(UNET_CARDIO, map_location=device))
    m1.eval()
    print("  ✓ U-Net cardio   (JSRT+VinDr Dice=0.8547)")

    m2 = UNet().to(device)
    m2.load_state_dict(torch.load(UNET_CALCIF, map_location=device))
    m2.eval()
    print("  ✓ U-Net calcif   (VinDr Dice=0.784)")

    m3 = EfficientNetCNN(num_classes=3).to(device)
    m3.load_state_dict(torch.load(CALCIF_MODEL, map_location=device), strict=False)
    m3.eval()
    print("  ✓ CNN calcif     (VinDr Acc=0.756)")

    m4 = EfficientNetCNN(num_classes=3).to(device)
    m4.load_state_dict(torch.load(RISK_MODEL, map_location=device))
    m4.eval()
    print("  ✓ CNN risque v2  (NIH AUC=0.895)")

    return m1, m2, m3, m4


# ============================================================
# BUG #2 CORRIGÉ — Chargement CSV centralisé
# ============================================================
def charger_csv(csv_path):
    """
    Charge le CSV NIH UNE SEULE FOIS et l'indexe par 'Image Index'.
    Retourne un DataFrame indexé prêt pour lookup O(1).

    AVANT (bug) : pd.read_csv() appelé dans analyser_radio() à chaque image
    APRÈS       : appelé une seule fois ici, passé en paramètre df_meta
    """
    print(f"Chargement CSV : {csv_path} ...")
    df = pd.read_csv(csv_path)
    df = df.set_index('Image Index')   # ← index pour lookup O(1)
    print(f"  ✓ {len(df)} entrées chargées en mémoire")
    return df


# ============================================================
# 3. CALCUL CTR — INCHANGÉ
# ============================================================
def calculer_ctr(mask_cardio, spacing_corrige, position):
    h, w = mask_cardio.shape
    contours, _ = cv2.findContours(
        mask_cardio.astype(np.uint8),
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_SIMPLE
    )
    heart_px = 0
    if contours:
        c = max(contours, key=cv2.contourArea)
        x, y, w_box, h_box = cv2.boundingRect(c)
        cx = x + w_box // 2
        cy = y + h_box // 2
        if w * 0.20 < cx < w * 0.80 and h * 0.20 < cy < h * 0.80:
            heart_px = w_box

    thorax_px = int(w * 0.75)
    heart_cm  = (heart_px  * spacing_corrige) / 10
    thorax_cm = (thorax_px * spacing_corrige) / 10
    ctr_raw   = heart_cm / thorax_cm if thorax_cm > 0 and heart_px > 0 else 0
    ctr_corrige = ctr_raw - 0.05 if 'AP' in str(position).upper() else ctr_raw

    return ctr_corrige, heart_cm, thorax_cm, heart_px


# ============================================================
# 4. PIPELINE COMPLET — BUG #2 + #5 + #6 CORRIGÉS
# ============================================================
def analyser_radio(img_path, m_cardio, m_calcif_seg, m_calcif_cls, m_risk,
                   df_meta=None,      # ← NOUVEAU : DataFrame déjà chargé (Bug #2)
                   csv_path=None,     # ← Conservé pour compatibilité usage simple
                   image_name=None):
    """
    CORRECTIONS :
    - df_meta   : DataFrame NIH pré-chargé (évite pd.read_csv à chaque appel)
    - csv_path  : toujours accepté pour usage simple (sera ignoré si df_meta fourni)
    - ctr_warning : affiché si CTR non calibré (Bug #5)
    - T512/T224 : constantes globales au lieu de recréation (Bug #6)
    """

    # --- Chargement image ---
    img_pil  = Image.open(img_path).convert('L')
    img_orig = img_pil.size
    img_np   = np.array(img_pil.resize((512, 512)))

    # --- Métadonnées ---
    spacing    = 0.143
    position   = 'PA'
    age        = 'N/A'
    gender     = 'N/A'
    label_reel = 'N/A'
    img_orig_w = img_orig[0]
    ctr_warning = ''   # ← BUG #5

    # ── BUG #2 CORRIGÉ : lookup O(1) dans df_meta déjà chargé ──────
    if df_meta is not None and image_name is not None:
        # Cas normal Phase 4 : df_meta fourni par tester_pipeline()
        if image_name in df_meta.index:
            row        = df_meta.loc[image_name]
            spacing    = float(row.get('OriginalImagePixelSpacing[x', 0.143))
            position   = str(row.get('View Position', 'PA'))
            age        = row.get('Patient Age', 'N/A')
            gender     = row.get('Patient Gender', 'N/A')
            label_reel = row.get('Finding Labels', 'N/A')
            img_orig_w = float(row.get('OriginalImage[Width', img_orig[0]))
        else:
            print(f"  ⚠ Image non trouvée dans df_meta : {image_name}")

    elif csv_path and image_name:
        # Fallback compatibilité : si df_meta non fourni, relit le CSV
        # (usage simple image par image — pas la Phase 4)
        try:
            df  = pd.read_csv(csv_path)
            row = df[df['Image Index'] == image_name]
            if not row.empty:
                row        = row.iloc[0]
                spacing    = float(row.get('OriginalImagePixelSpacing[x', 0.143))
                position   = str(row.get('View Position', 'PA'))
                age        = row.get('Patient Age', 'N/A')
                gender     = row.get('Patient Gender', 'N/A')
                label_reel = row.get('Finding Labels', 'N/A')
                img_orig_w = float(row.get('OriginalImage[Width', img_orig[0]))
        except Exception as e:
            print(f"  Avertissement CSV : {e}")

    # Correction spacing
    if df_meta is None and csv_path is None:
        # ── BUG #5 CORRIGÉ : avertissement explicite CTR non calibré ──
        spacing_corrige = 0.143
        scale_factor    = 1.0
        ctr_warning     = '⚠ CTR non calibré (image externe sans CSV)'
    else:
        scale_factor    = img_orig_w / 512.0
        spacing_corrige = spacing * scale_factor

    # ── BUG #6 CORRIGÉ : utilisation des constantes globales T512/T224 ──
    inp512 = T512(img_pil).unsqueeze(0).to(device)
    inp224 = T224(img_pil).unsqueeze(0).to(device)

    # ── MODULE 1 : U-Net cardio → CTR ──────────────────────────────
    with torch.no_grad():
        mask_cardio = (m_cardio(inp512).cpu().squeeze().numpy() > 0.7).astype(np.uint8)

    ctr_corrige, heart_cm, thorax_cm, heart_px = calculer_ctr(
        mask_cardio, spacing_corrige, position
    )
    cardio_flag = ctr_corrige > 0.50

    # ── MODULE 2 : U-Net calcif → masque ───────────────────────────
    with torch.no_grad():
        mask_calcif = (m_calcif_seg(inp512).cpu().squeeze().numpy() > 0.5).astype(np.uint8)

    calcif_pct = float(mask_calcif.sum()) / mask_calcif.size * 100

    # ── MODULE 3 : CNN calcif → score ──────────────────────────────
    with torch.no_grad():
        out3  = m_calcif_cls(inp224)
        prob3 = torch.softmax(out3, dim=1).cpu().numpy()[0]

    calcif_cls_label = CATEGORIES_CALCIF[np.argmax(prob3)]
    calcif_cls_score = float(prob3[1]) + float(prob3[2])

    # ── MODULE 4 : CNN risque → congestion + Grad-CAM ──────────────
    cam, out4  = m_risk.get_gradcam(inp224)
    prob4      = torch.softmax(out4, dim=1).detach().cpu().numpy()[0]
    risk_label = CATEGORIES_RISK[np.argmax(prob4)]
    prob_edema  = float(prob4[2])
    prob_cardio = float(prob4[1])

    # ── FUSION : Score risque global ───────────────────────────────
    s_ctr    = min(max((ctr_corrige - 0.40) / 0.20, 0), 1)
    s_calcif = min(calcif_cls_score, 1)
    s_cong   = prob_edema
    risk_global = round((0.50 * s_ctr + 0.25 * s_calcif + 0.25 * s_cong) * 100, 1)

    if risk_global > 60:
        niveau, couleur = 'ÉLEVÉ',  'red'
    elif risk_global > 30:
        niveau, couleur = 'MODÉRÉ', 'orange'
    else:
        niveau, couleur = 'FAIBLE', 'green'

    # ── RAPPORT CONSOLE ────────────────────────────────────────────
    nom = image_name or os.path.basename(img_path)
    print("\n" + "="*60)
    print(f"  RAPPORT CARDIOVASCULAIRE")
    print(f"  Image    : {nom}")
    print(f"  Label    : {label_reel}")
    print(f"  Patient  : Âge={age} | Sexe={gender} | Vue={position}")
    print(f"  Spacing  : {spacing:.3f} → corrigé {spacing_corrige:.3f} mm/px")
    print("="*60)
    print(f"\n  MODULE 1 — TAILLE CARDIAQUE")
    # BUG #5 : ctr_warning affiché si non calibré
    print(f"    CTR corrigé : {ctr_corrige:.3f}  "
          f"{'⚠ Cardiomégalie' if cardio_flag else '✓ Normal'}  {ctr_warning}")
    print(f"    Cœur        : {heart_cm:.2f} cm")
    print(f"    Thorax      : {thorax_cm:.2f} cm")
    print(f"\n  MODULE 2+3 — CALCIFICATIONS")
    print(f"    Masque U-Net : {calcif_pct:.2f}%")
    print(f"    CNN classify : {calcif_cls_label} ({calcif_cls_score*100:.1f}%)")
    print(f"\n  MODULE 4 — CONGESTION PULMONAIRE")
    print(f"    Prob Edema   : {prob_edema*100:.1f}%")
    print(f"    Prob Cardio  : {prob_cardio*100:.1f}%")
    print(f"    Diagnostic   : {risk_label}")
    print(f"\n{'─'*60}")
    print(f"  RISQUE GLOBAL : {risk_global}% [{niveau}]")
    print(f"    CTR (50%)           : {round(s_ctr*50, 1)}/50")
    print(f"    Calcifications (25%): {round(s_calcif*25, 1)}/25")
    print(f"    Congestion (25%)    : {round(s_cong*25, 1)}/25")
    print("="*60)

    # ── VISUALISATION 6 PANNEAUX ───────────────────────────────────
    fig = plt.figure(figsize=(22, 8))
    gs  = gridspec.GridSpec(2, 4, figure=fig)

    ax0 = fig.add_subplot(gs[:, 0])
    ax0.imshow(img_np, cmap='gray')
    ax0.set_title('Radio originale', fontweight='bold')
    ax0.axis('off')

    ax1 = fig.add_subplot(gs[0, 1])
    ax1.imshow(img_np, cmap='gray')
    mask_c_r = cv2.resize(mask_cardio.astype(np.uint8),
                          (img_np.shape[1], img_np.shape[0]))
    ax1.imshow(mask_c_r, cmap='Reds', alpha=0.4)
    color_ctr = 'red' if cardio_flag else 'green'
    ax1.set_title(
        f"Cœur — CTR={ctr_corrige:.2f}\n"
        f"{'⚠ Cardiomégalie' if cardio_flag else '✓ Normal'}",
        fontweight='bold', color=color_ctr
    )
    ax1.axis('off')

    ax2 = fig.add_subplot(gs[0, 2])
    ax2.imshow(img_np, cmap='gray')
    mask_k_r = cv2.resize(mask_calcif.astype(np.uint8),
                          (img_np.shape[1], img_np.shape[0]))
    ax2.imshow(mask_k_r, cmap='hot', alpha=0.5)
    ax2.set_title(
        f"Calcifications\n{calcif_pct:.2f}% — {calcif_cls_label}",
        fontweight='bold',
        color='red' if calcif_cls_score > 0.5 else 'green'
    )
    ax2.axis('off')

    ax3 = fig.add_subplot(gs[0, 3])
    if cam is not None:
        img_rgb = cv2.cvtColor(img_np, cv2.COLOR_GRAY2RGB)
        cam_r   = cv2.resize(cam, (img_np.shape[1], img_np.shape[0]))
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_r), cv2.COLORMAP_JET)
        overlay = cv2.addWeighted(
            img_rgb, 0.5,
            cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB), 0.5, 0
        )
        ax3.imshow(overlay)
    ax3.set_title(f'Grad-CAM\n→ {risk_label}', fontweight='bold')
    ax3.axis('off')

    ax4 = fig.add_subplot(gs[1, 1])
    ax4.barh(CATEGORIES_RISK, prob4 * 100, color=['green', 'red', 'orange'])
    ax4.set_xlim(0, 100); ax4.set_xlabel('%')
    ax4.set_title('CNN Risque (M4)', fontweight='bold')
    for i, v in enumerate(prob4 * 100):
        ax4.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=9)

    ax5 = fig.add_subplot(gs[1, 2])
    ax5.barh(CATEGORIES_CALCIF, prob3 * 100, color=['green', 'orange', 'red'])
    ax5.set_xlim(0, 100); ax5.set_xlabel('%')
    ax5.set_title('CNN Calcif (M3)', fontweight='bold')
    for i, v in enumerate(prob3 * 100):
        ax5.text(v + 1, i, f'{v:.1f}%', va='center', fontsize=9)

    ax6 = fig.add_subplot(gs[1, 3])
    contributions = {
        'CTR (50%)':        round(s_ctr * 50, 1),
        'Calcif (25%)':     round(s_calcif * 25, 1),
        'Congestion (25%)': round(s_cong * 25, 1)
    }
    bars = ax6.barh(list(contributions.keys()),
                    list(contributions.values()),
                    color=['red', 'orange', 'blue'])
    ax6.set_xlim(0, 55)
    ax6.set_title(f'Risque global : {risk_global}%\n[{niveau}]',
                  fontweight='bold', color=couleur)
    for bar, v in zip(bars, contributions.values()):
        ax6.text(bar.get_width() + 0.5,
                 bar.get_y() + bar.get_height() / 2,
                 str(v), va='center', fontsize=9)

    plt.suptitle(
        f'Analyse Cardiovasculaire — {nom}  |  Label réel: {label_reel}',
        fontsize=13, fontweight='bold'
    )
    plt.tight_layout()
    fname = f"rapport_{nom.replace('.png','').replace('.jpg','')}.png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Rapport sauvegardé : {fname}")

    return {
        'image':        nom,
        'label_reel':   label_reel,
        'ctr':          round(ctr_corrige, 3),
        'cardio_flag':  cardio_flag,
        'calcif_pct':   round(calcif_pct, 2),
        'calcif_label': calcif_cls_label,
        'prob_edema':   round(prob_edema, 3),
        'prob_cardio':  round(prob_cardio, 3),
        'risk_label':   risk_label,
        'risk_global':  risk_global,
        'niveau':       niveau
    }


# ============================================================
# 5. TEST SUR PLUSIEURS IMAGES — BUG #2 CORRIGÉ
# ============================================================
def tester_pipeline(m_cardio, m_calcif_seg, m_calcif_cls, m_risk,
                    categories=['Cardiomegaly', 'Normal', 'Edema'],
                    n_par_categorie=2):
    """
    BUG #2 CORRIGÉ :
    - df_meta chargé UNE SEULE FOIS via charger_csv() au début
    - Passé à analyser_radio() via df_meta au lieu de csv_path
    - Gain : 500× plus rapide pour Phase 4 sur 500 images
    """
    # ── Chargement CSV une seule fois ─────────────────────────────
    df_meta   = charger_csv(CSV_PATH)          # ← UNE SEULE LECTURE
    df_filtre = pd.read_csv(CSV_PATH)          # ← pour filtrer par catégorie
    resultats = []

    for cat in categories:
        if cat == 'Normal':
            subset = df_filtre[
                df_filtre['Finding Labels'] == 'No Finding'
            ].head(n_par_categorie)
        else:
            subset = df_filtre[
                df_filtre['Finding Labels'].str.contains(cat)
            ].head(n_par_categorie)

        for _, row in subset.iterrows():
            img_name = row['Image Index']
            img_path = os.path.join(IMG_DIR, img_name)

            if not os.path.exists(img_path):
                print(f"Image non trouvée : {img_name}")
                continue

            print(f"\nTEST : {img_name} — Label : {row['Finding Labels']}")
            r = analyser_radio(
                img_path     = img_path,
                m_cardio     = m_cardio,
                m_calcif_seg = m_calcif_seg,
                m_calcif_cls = m_calcif_cls,
                m_risk       = m_risk,
                df_meta      = df_meta,    # ← df pré-chargé (Bug #2)
                image_name   = img_name
            )
            r['label_reel'] = row['Finding Labels']
            resultats.append(r)

    if resultats:
        print("\n" + "="*85)
        print("TABLEAU RÉCAPITULATIF")
        print("="*85)
        print(f"{'Image':<25} {'Label réel':<25} {'CTR':<8} "
              f"{'Calcif%':<10} {'Risque%':<10} {'Niveau'}")
        print("-"*85)
        for r in resultats:
            print(f"{r['image']:<25} {r['label_reel']:<25} "
                  f"{r['ctr']:<8.3f} {r['calcif_pct']:<10.2f} "
                  f"{r['risk_global']:<10.1f} {r['niveau']}")

        pd.DataFrame(resultats).to_csv('resultats_pipeline.csv', index=False)
        print("\nRésultats sauvegardés : resultats_pipeline.csv")

    return resultats


# ============================================================
# EXÉCUTION
# ============================================================
if __name__ == "__main__":

    # 1. Charger les 4 modèles
    m_cardio, m_calcif_seg, m_calcif_cls, m_risk = charger_modeles()

    # ── Charger le CSV UNE SEULE FOIS pour les tests NIH ──────────
    df_meta = charger_csv(CSV_PATH)   # ← BUG #2 CORRIGÉ

    # 2. TEST image NIH connue
    r = analyser_radio(
        img_path     = f'{IMG_DIR}/00000001_000.png',
        m_cardio     = m_cardio,
        m_calcif_seg = m_calcif_seg,
        m_calcif_cls = m_calcif_cls,
        m_risk       = m_risk,
        df_meta      = df_meta,        # ← lookup O(1), pas de relecture
        image_name   = '00000001_000.png'
    )

    # 3. TEST im1 externe (sans CSV — ctr_warning s'affichera)
    r1 = analyser_radio(
        img_path     = '/kaggle/input/datasets/abdallahouahmane/images/im1-Chest-X-ray.jpg',
        m_cardio     = m_cardio,
        m_calcif_seg = m_calcif_seg,
        m_calcif_cls = m_calcif_cls,
        m_risk       = m_risk,
        df_meta      = None,
        image_name   = 'im1-Chest-X-ray.jpg'
    )

    # 4. TEST im2 externe (sans CSV — ctr_warning s'affichera)
    r2 = analyser_radio(
        img_path     = '/kaggle/input/datasets/abdallahouahmane/images/im3.png',
        m_cardio     = m_cardio,
        m_calcif_seg = m_calcif_seg,
        m_calcif_cls = m_calcif_cls,
        m_risk       = m_risk,
        df_meta      = None,
        image_name   = 'im3.png'
    )

    # 5. Résumé comparatif
    print("\n" + "="*65)
    print("COMPARAISON DES 3 TESTS")
    print("="*65)
    print(f"{'Image':<25} {'CTR':<8} {'Cardio?':<12} {'Risque%':<10} {'Niveau'}")
    print("-"*65)
    for res, nom in [(r, '00000001_000.png'),
                     (r1,'im1-Chest-X-ray.jpg'),
                     (r2,'im3.png')]:
        print(f"{nom:<25} {res['ctr']:<8.3f} "
              f"{'⚠ OUI' if res['cardio_flag'] else '✓ NON':<12} "
              f"{res['risk_global']:<10.1f} {res['niveau']}")